<a href="https://colab.research.google.com/github/MBR4V0/Python/blob/main/Aplication_RAG_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install openAI

In [ ]:
# ==========================================================
# INSTALAÇÃO
# ==========================================================

!pip install -q faiss-cpu sentence-transformers pandas numpy

# ==========================================================
# IMPORTS
# ==========================================================

import re
import uuid
import faiss
import pandas as pd
import numpy as np

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

# ==========================================================
# CONFIGURAÇÃO
# ==========================================================

CSV_PATH = "erros.csv"

HISTORICO_PATH = "historico.csv"

EMBEDDING_MODEL = "intfloat/multilingual-e5-base"

RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

TOP_K_FAISS = 20

TOP_K_FINAL = 3

MIN_SCORE = 0.50

# ==========================================================
# NORMALIZAÇÃO
# ==========================================================

def normalizar(texto):

    texto = str(texto).lower()

    texto = re.sub(
        r'[^a-z0-9áàâãéêíóôõúç ]',
        '',
        texto
    )

    texto = re.sub(
        r'\s+',
        ' ',
        texto
    )

    return texto.strip()

# ==========================================================
# CARREGAR BASE
# ==========================================================

print("Carregando base...")

df = pd.read_csv(
    CSV_PATH,
    encoding="latin1"
)

df["erro"] = df["erro"].fillna("")

df["causa"] = df["causa"].fillna("")

df["solucao"] = df["solucao"].fillna("")

df["texto_busca"] = (
    df["erro"]
    .apply(normalizar)
)

# ==========================================================
# MODELOS
# ==========================================================

print("Carregando Embedding Model...")

embedder = SentenceTransformer(
    EMBEDDING_MODEL
)

print("Carregando CrossEncoder...")

reranker = CrossEncoder(
    RERANK_MODEL
)

# ==========================================================
# EMBEDDINGS
# ==========================================================

print("Gerando embeddings...")

embeddings = embedder.encode(
    df["texto_busca"].tolist(),
    normalize_embeddings=True,
    convert_to_numpy=True
)

# ==========================================================
# FAISS
# ==========================================================

dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)

index.add(
    embeddings.astype("float32")
)

# ==========================================================
# HISTÓRICO
# ==========================================================

def carregar_historico():

    try:

        return pd.read_csv(
            HISTORICO_PATH
        )

    except:

        return pd.DataFrame(
            columns=[
                "id",
                "pergunta",
                "erro",
                "causa",
                "solucao",
                "score",
                "status"
            ]
        )

# ==========================================================
# ESTATÍSTICAS
# ==========================================================

def calcular_confianca(solucao):

    hist = carregar_historico()

    registros = hist[
        hist["solucao"] == solucao
    ]

    aprovados = len(
        registros[
            registros["status"] == "APROVADO"
        ]
    )

    rejeitados = len(
        registros[
            registros["status"] == "REJEITADO"
        ]
    )

    total = aprovados + rejeitados

    if total == 0:

        return 0.5

    return aprovados / total

# ==========================================================
# SALVAR FEEDBACK
# ==========================================================

def salvar_feedback(
    pergunta,
    resultado,
    status
):

    hist = carregar_historico()

    nova_linha = {

        "id":
        str(uuid.uuid4()),

        "pergunta":
        pergunta,

        "erro":
        resultado["erro"],

        "causa":
        resultado["causa"],

        "solucao":
        resultado["solucao"],

        "score":
        resultado["score_final"],

        "status":
        status
    }

    hist = pd.concat(
        [
            hist,
            pd.DataFrame(
                [nova_linha]
            )
        ],
        ignore_index=True
    )

    hist.to_csv(
        HISTORICO_PATH,
        index=False
    )

# ==========================================================
# BUSCA
# ==========================================================

def buscar(pergunta):

    pergunta_norm = normalizar(
        pergunta
    )

    query_emb = embedder.encode(
        [pergunta_norm],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    scores, idxs = index.search(
        query_emb.astype("float32"),
        TOP_K_FAISS
    )

    candidatos = []

    for score, idx in zip(
        scores[0],
        idxs[0]
    ):

        if idx == -1:
            continue

        row = df.iloc[idx]

        candidatos.append({

            "erro":
            row["erro"],

            "causa":
            row["causa"],

            "solucao":
            row["solucao"],

            "score_faiss":
            float(score)
        })

    if not candidatos:

        return []

    # ======================================================
    # CROSS ENCODER
    # ======================================================

    pares = [

        (
            pergunta,
            c["erro"]
        )

        for c in candidatos
    ]

    cross_scores = reranker.predict(
        pares
    )

    resultados = []

    for c, cross in zip(
        candidatos,
        cross_scores
    ):

        taxa = calcular_confianca(
            c["solucao"]
        )

        score_final = (

            0.50 * c["score_faiss"] +

            0.30 * float(cross) +

            0.20 * taxa

        )

        c["cross_score"] = float(cross)

        c["taxa_sucesso"] = taxa

        c["score_final"] = score_final

        resultados.append(c)

    resultados.sort(

        key=lambda x:
        x["score_final"],

        reverse=True
    )

    return resultados[
        :TOP_K_FINAL
    ]

# ==========================================================
# CHAT
# ==========================================================

def rodar_chat():

    print("\n============================")
    print("ERP AI RAG v2")
    print("============================\n")

    while True:

        pergunta = input(
            "\nDigite o erro: "
        )

        if pergunta.lower() == "sair":

            break

        resultados = buscar(
            pergunta
        )

        if not resultados:

            print(
                "\nNenhum resultado encontrado."
            )

            continue

        encontrou = False

        for r in resultados:

            if r["score_final"] < MIN_SCORE:

                continue

            print("\n--------------------------------")

            print(
                f"\nScore Final: "
                f"{r['score_final']:.3f}"
            )

            print(
                f"Taxa de sucesso: "
                f"{r['taxa_sucesso']:.1%}"
            )

            print(
                f"\nErro:\n{r['erro']}"
            )

            print(
                f"\nCausa:\n{r['causa']}"
            )

            print(
                f"\nSolução:\n{r['solucao']}"
            )

            resposta = input(
                "\nFuncionou? (sim/não): "
            ).lower()

            if resposta == "sim":

                salvar_feedback(
                    pergunta,
                    r,
                    "APROVADO"
                )

                print(
                    "\n✅ Solução confirmada."
                )

                encontrou = True

                break

            else:

                salvar_feedback(
                    pergunta,
                    r,
                    "REJEITADO"
                )

                print(
                    "\n⚠️ Tentando próxima..."
                )

        if not encontrou:

            print(
                "\nNenhuma solução aprovada."
            )

# ==========================================================
# EXECUTAR
# ==========================================================

rodar_chat()

Carregando base...
Carregando Embedding Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Gerando embeddings...

ERP AI RAG v2


Digite o erro: Falha de Nota Fiscal

Nenhuma solução aprovada.

Digite o erro: Erro em nota Fiscal

--------------------------------

Score Final: 1.543
Taxa de sucesso: 50.0%

Erro:
Erro SPED fiscal

Causa:
Arquivo corrompido

Solução:
Gerar novo SPED

Funcionou? (sim/não): Sim

✅ Solução confirmada.

Digite o erro: Erro em nota Fiscal

--------------------------------

Score Final: 1.643
Taxa de sucesso: 100.0%

Erro:
Erro SPED fiscal

Causa:
Arquivo corrompido

Solução:
Gerar novo SPED

Funcionou? (sim/não): Não

⚠️ Tentando próxima...

Nenhuma solução aprovada.

Digite o erro: Erro em nota Fiscal

--------------------------------

Score Final: 1.543
Taxa de sucesso: 50.0%

Erro:
Erro SPED fiscal

Causa:
Arquivo corrompido

Solução:
Gerar novo SPED
